# Developer Career Intelligence Platform - 01 Data Collection & Ingestion

This notebook demonstrates the end-to-end ingestion pipeline:
1. Ingesting public GitHub user metadata, repositories, languages, and commit logs.
2. Applying disk-based caching and exponential backoff to handle rate limits.
3. Persisting normalized entities into the SQLite data warehouse.

In [1]:
import os
import sys
sys.path.append('..')

from database.database import DatabaseManager
from ingestion.github_client import GitHubAPIClient
from ingestion.collectors import DeveloperDataCollector
from ingestion.sample_profiles import list_sample_profiles, seed_sample_profiles

# Initialize Database
db = DatabaseManager('../data/developer_intelligence.db')
print("Database initialized at:", db.db_path)

In [2]:
# Pre-seed curated developer profiles (offline mode guarantee)
seed_sample_profiles(db)
users_df = db.get_all_users()
print(f"Total ingested profiles: {len(users_df)}")
users_df[['id', 'username', 'name', 'public_repos', 'followers']]

In [3]:
# Inspect Repositories and Languages for Alex Vance (Data Scientist)
user_id = users_df.loc[users_df['username'] == 'alex-datascientist', 'id'].iloc[0]
repos = db.get_repositories(user_id)
langs = db.get_languages(user_id)

print(f"Repositories count: {len(repos)}")
print(f"Language breakdown records: {len(langs)}")
repos[['repo_name', 'primary_language', 'stargazers_count', 'size_kb']].head()